# Notebook 08: Lexical Content-Based Baseline — TF-IDF User Profile Similarity

## 1. Formulation
Each article $i$ is represented by a TF-IDF vector $\mathbf{t}_i \in \mathbb{R}^V$ over title + summary tokens. Each user's local profile is the dwell-weighted centroid $\mathbf{p}_u = \frac{\sum_{i \in \mathcal{H}_u} w_{ui} \mathbf{t}_i}{\sum w_{ui}}$, scored via cosine similarity $\cos(\mathbf{p}_u, \mathbf{t}_j)$.

In [1]:
import os, sys, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Resolve repository root deterministically from ml/notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DATA_DIR = os.path.join(REPO_ROOT, "ml", "data", "processed")
ARTIFACTS_DIR = os.path.join(REPO_ROOT, "ml", "artifacts")
VIZ_DIR = os.path.join(REPO_ROOT, "ml", "visualizations")
plt.rcParams["figure.dpi"] = 110
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from ml.src.data.dataset import RecSysDataPipeline
from ml.src.evaluation.metrics import hit_rate_at_k, ndcg_at_k, mrr

pipeline = RecSysDataPipeline(
    articles_path=os.path.join(DATA_DIR, "articles.json"),
    interactions_path=os.path.join(DATA_DIR, "interactions.csv"),
    embeddings_path=os.path.join(DATA_DIR, "article_embeddings.npy"),
)

corpus_texts = [f"{a.get('title', '')} {a.get('category', '')} {a.get('subcategory', '')} {a.get('summary', '')}" for a in pipeline.articles]
vec = TfidfVectorizer(max_features=3000, stop_words="english")
tfidf_mat = vec.fit_transform(corpus_texts)

# Build per-user centroid strictly from training split
user_profiles = {}
for uid, grp in pipeline.train_df.groupby("user_id"):
    idxs = [pipeline.item2idx[a] for a in grp["article_id"] if a in pipeline.item2idx]
    if idxs:
        user_profiles[uid] = np.asarray(tfidf_mat[idxs].mean(axis=0))

hr_l, ndcg_l, mrr_l = [], [], []
for _, row in pipeline.test_df.iterrows():
    uid = row["user_id"]
    u_idx = pipeline.user2idx[uid]
    pos_idx = pipeline.item2idx[row["article_id"]]
    user_pos = pipeline.user_positive_items[u_idx]
    cands = [pos_idx]
    while len(cands) < 100:
        c = random.randint(0, pipeline.num_items - 1)
        if c not in user_pos and c not in cands:
            cands.append(c)
    scores = cosine_similarity(user_profiles[uid], tfidf_mat[cands]).ravel()
    ranked = [cands[i] for i in np.argsort(-scores)]
    hr_l.append(hit_rate_at_k(pos_idx, ranked, k=10))
    ndcg_l.append(ndcg_at_k(pos_idx, ranked, k=10))
    mrr_l.append(mrr(pos_idx, ranked))

print(f"TF-IDF Vocabulary Size : {len(vec.vocabulary_):,}")
print(f"TF-IDF Content Baseline | HR@10 = {np.mean(hr_l):.4f} | NDCG@10 = {np.mean(ndcg_l):.4f} | MRR = {np.mean(mrr_l):.4f}")

TF-IDF Vocabulary Size : 598
TF-IDF Content Baseline | HR@10 = 0.2900 | NDCG@10 = 0.1316 | MRR = 0.1108
